In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)
AV_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
URL = "https://www.alphavantage.co/query"
symbol = "NVDA"

def av(params):
    r = requests.get(URL, params={**params, "apikey": AV_KEY}, timeout=30)
    r.raise_for_status()
    data = r.json()
    msg = data.get("Information") or data.get("Note") or data.get("Error Message")
    return data, msg

# 1) Intraday 1-minute bars (what the video uses; premium on most free keys)
data, msg = av({"function": "TIME_SERIES_INTRADAY", "symbol": symbol,
                "interval": "1min", "outputsize": "compact"})

if not msg:
    ts_key = [k for k in data if "Time Series" in k][0]
    df = pd.DataFrame(data[ts_key]).T
    df.columns = [c.split(". ", 1)[1] for c in df.columns]   # "1. open" -> "open"
    df.index = pd.to_datetime(df.index)
    df = df.astype(float).sort_index()
    print(f"{symbol} intraday (1min), {len(df)} bars:")
    print(df.tail(10))
else:
    print("Intraday not available on this key:")
    print(" ", msg[:200], "\n")

    # 2) Fallback: latest quote (free)
    data, msg = av({"function": "GLOBAL_QUOTE", "symbol": symbol})
    if msg:
        print(msg[:200])
    else:
        quote = pd.DataFrame([data["Global Quote"]])
        quote.columns = [c.split(". ", 1)[1] for c in quote.columns]
        print(f"{symbol} latest quote:")
        print(quote.T)

Intraday not available on this key:
  Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints 

Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premium plans at https://www.alphavantage


In [2]:
import time
time.sleep(1.5)   # respect the 1 request/second limit

data, msg = av({"function": "GLOBAL_QUOTE", "symbol": "NVDA"})
if msg:
    print(msg[:200])
else:
    quote = pd.DataFrame([data["Global Quote"]])
    quote.columns = [c.split(". ", 1)[1] for c in quote.columns]
    print(quote.T)

                             0
symbol                    NVDA
open                  144.1300
high                  150.3700
low                   141.1700
price                 149.3000
volume               359407890
latest trading day  2026-11-25
previous close        155.3800
change                 -6.0800
change percent        -3.9130%


In [3]:
import yfinance as yf

# "Real-time" 1-minute bars (delayed ~15 min during market hours)
df = yf.download("NVDA", period="1d", interval="1m", multi_level_index=False, progress=False)
df.index = df.index.tz_convert("America/New_York")
print(f"NVDA 1-minute bars: {len(df)} | latest bar: {df.index[-1]}")
print(df.tail(10))

NVDA 1-minute bars: 390 | latest bar: 2026-11-25 15:59:00-05:00
                                Close        High         Low        Open  \
Datetime                                                                    
2026-11-25 15:50:00-05:00  149.140106  149.195007  148.529999  148.535004   
2026-11-25 15:51:00-05:00  149.289993  149.399994  149.130005  149.160706   
2026-11-25 15:52:00-05:00  149.089996  149.455002  149.050003  149.309998   
2026-11-25 15:53:00-05:00  149.050003  149.149994  148.979996  149.090103   
2026-11-25 15:54:00-05:00  149.009995  149.130005  148.804993  149.059998   
2026-11-25 15:55:00-05:00  149.110001  149.169998  148.550003  149.014999   
2026-11-25 15:56:00-05:00  149.184998  149.315002  149.024994  149.100006   
2026-11-25 15:57:00-05:00  149.212494  149.240005  148.990005  149.190002   
2026-11-25 15:58:00-05:00  149.300003  149.419998  149.050003  149.220001   
2026-11-25 15:59:00-05:00  149.289993  150.369995  149.089996  149.300003   

           